# Notebook 13 — Pivot and Unpivot

**Dataset:** `samples.bakehouse.sales_transactions` + `samples.tpch.orders` + `samples.wanderbricks.bookings` + `properties`  
**Difficulty:** Medium  
**Topics:** `pivot`, `unpivot (stack)`, `crosstab`, `groupBy + agg with pivot`


In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder.getOrCreate()

transactions = spark.table("samples.bakehouse.sales_transactions")
orders       = spark.table("samples.tpch.orders")
bookings     = spark.table("samples.wanderbricks.bookings")
properties   = spark.table("samples.wanderbricks.properties")


## Problem 1 — Pivot Payment Methods to Columns

On `sales_transactions`, groupBy `product`, pivot on `paymentMethod` to get total revenue per product per payment method as separate columns.  
**Columns:** `product`, `<each_payment_method_as_column>`


In [ ]:
result_1 = None
# YOUR CODE HERE


In [ ]:
# ── Tests for Problem 1 ─────────────────
assert result_1 is not None, "result_1 is None"
assert hasattr(result_1, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_1.columns]
assert 'product' in cols, "Missing: product"
cnt = result_1.count()
assert cnt > 0, "Got 0 rows"
# Should have more than 1 column (product + at least one payment method)
assert len(result_1.columns) > 1, "Expected multiple columns after pivot"
# Number of rows = number of distinct products
distinct_products = transactions.select("product").distinct().count()
assert cnt == distinct_products, f"Expected {distinct_products} rows, got {cnt}"
print(f"Problem 1 passed ✓  ({cnt} rows, {len(result_1.columns)} columns)")


## Problem 2 — Pivot Order Status by Year

On `tpch.orders`, extract year from `o_orderdate`. Pivot on `o_orderstatus` to get order count per year per status as columns.  
**Columns:** `year`, `F` (Fulfilled count), `O` (Open count), `P` (Partial count)


In [ ]:
result_2 = None
# YOUR CODE HERE


In [ ]:
# ── Tests for Problem 2 ─────────────────
assert result_2 is not None, "result_2 is None"
assert hasattr(result_2, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_2.columns]
assert 'year' in cols, "Missing: year"
cnt = result_2.count()
assert cnt > 0, "Got 0 rows"
# Should have status columns F, O, P
assert 'f' in cols or 'o' in cols, "Expected order status columns after pivot"
# Years should be numeric/valid
assert result_2.agg(F.min("year")).collect()[0][0] >= 1990, "Year values look invalid"
print(f"Problem 2 passed ✓  ({cnt} rows)")


## Problem 3 — Pivot Booking Status by Property Type

Join `wanderbricks.bookings` with `properties` on `property_id`. Pivot on `status` to show count per property_type per status.  
**Columns:** `property_type`, `<status_columns>`


In [ ]:
result_3 = None
# YOUR CODE HERE


In [ ]:
# ── Tests for Problem 3 ─────────────────
assert result_3 is not None, "result_3 is None"
assert hasattr(result_3, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_3.columns]
assert 'property_type' in cols, "Missing: property_type"
cnt = result_3.count()
assert cnt > 0, "Got 0 rows"
# Should have more than just property_type column
assert len(result_3.columns) > 1, "Expected status columns after pivot"
# Number of rows should equal distinct property types
distinct_types = properties.select("property_type").distinct().count()
assert cnt == distinct_types, f"Expected {distinct_types} rows (distinct property_types), got {cnt}"
print(f"Problem 3 passed ✓  ({cnt} rows, {len(result_3.columns)} columns)")


## Problem 4 — Unpivot (stack) — Reverse a Pivot

Take the wide DataFrame from Problem 1 (or rebuild it). Use the `stack()` SQL expression to unpivot it back to long format.  
Pattern: `F.expr("stack(N, 'method1', method1, 'method2', method2, ...) as (payment_method, total_revenue)")`  
**Columns:** `product`, `payment_method`, `total_revenue`


In [ ]:
result_4 = None
# YOUR CODE HERE


In [ ]:
# ── Tests for Problem 4 ─────────────────
assert result_4 is not None, "result_4 is None"
assert hasattr(result_4, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_4.columns]
assert 'product' in cols, "Missing: product"
assert 'payment_method' in cols, "Missing: payment_method"
assert 'total_revenue' in cols, "Missing: total_revenue"
cnt = result_4.count()
assert cnt > 0, "Got 0 rows"
# Should have more rows than result_1 (wide -> long expansion)
assert cnt >= result_1.count(), "Unpivoted result should have >= rows than pivoted"
print(f"Problem 4 passed ✓  ({cnt} rows)")


## Problem 5 — Crosstab

Use `.crosstab("product", "paymentMethod")` on `sales_transactions` to generate a frequency count table.  
Note: crosstab column naming uses `product_paymentMethod` for the first column.  
Display the result.


In [ ]:
result_5 = None
# YOUR CODE HERE


In [ ]:
# ── Tests for Problem 5 ─────────────────
assert result_5 is not None, "result_5 is None"
assert hasattr(result_5, 'columns'), "Must be a Spark DataFrame"
cnt = result_5.count()
assert cnt > 0, "Got 0 rows"
# First column should be the cross-tab key column (product_paymentMethod)
first_col = result_5.columns[0].lower()
assert 'product' in first_col, f"First column should reference product, got: {first_col}"
# Should have multiple columns (one per payment method)
assert len(result_5.columns) > 1, "Expected crosstab to produce multiple columns"
print(f"Problem 5 passed ✓  ({cnt} rows, {len(result_5.columns)} columns)")


## Problem 6 — Multi-Level Pivot with Multiple Aggregations

Pivot `paymentMethod` on `sales_transactions` aggregating both `sum(totalPrice)` and `count(transactionID)`.  
Since pivot takes one agg at a time, do two separate pivots and join them, or use column renaming.  
**Columns:** `product`, `<method>_revenue`, `<method>_count` for each payment method


In [ ]:
result_6 = None
# YOUR CODE HERE


In [ ]:
# ── Tests for Problem 6 ─────────────────
assert result_6 is not None, "result_6 is None"
assert hasattr(result_6, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_6.columns]
assert 'product' in cols, "Missing: product"
cnt = result_6.count()
assert cnt > 0, "Got 0 rows"
# Should have both revenue and count columns (at least 2x payment methods + product)
assert len(result_6.columns) > 3, "Expected revenue and count columns for each payment method"
# Check at least one *_revenue or *_count column exists
revenue_cols = [c for c in cols if 'revenue' in c or 'count' in c]
assert len(revenue_cols) > 0, "Expected columns with 'revenue' or 'count' suffix"
print(f"Problem 6 passed ✓  ({cnt} rows, {len(result_6.columns)} columns)")


## Problem 7 — Rolling Pivot: Monthly Revenue per Product

Aggregate `sales_transactions` by `(month_label, product)` where `month_label` is formatted as `"YYYY-MM"`.  
Then pivot on `product` to see monthly revenue per product as columns.  
**Columns:** `month_label`, `<product_columns>`


In [ ]:
result_7 = None
# YOUR CODE HERE


In [ ]:
# ── Tests for Problem 7 ─────────────────
assert result_7 is not None, "result_7 is None"
assert hasattr(result_7, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_7.columns]
assert 'month_label' in cols, "Missing: month_label"
cnt = result_7.count()
assert cnt > 0, "Got 0 rows"
# Should have product columns beyond month_label
assert len(result_7.columns) > 1, "Expected product columns after pivot"
# month_label should follow YYYY-MM format (length 7)
bad_months = result_7.filter(F.length(F.col("month_label")) != 7)
assert bad_months.count() == 0, "month_label should be in YYYY-MM format"
print(f"Problem 7 passed ✓  ({cnt} rows, {len(result_7.columns)} columns)")
